In [ ]:
!pip install vk_api

Токен

In [ ]:
import vk_api
from vk_api.longpoll import VkLongPoll, VkEventType
from vk_api.keyboard import VkKeyboard, VkKeyboardColor
import sqlite3
import os
import json
import datetime
from google.colab import drive

drive.mount('/content/drive')
OBSIDIAN_FOLDER = '/content/drive/MyDrive/Obsidian_Bot'
if not os.path.exists(OBSIDIAN_FOLDER):
    os.makedirs(OBSIDIAN_FOLDER)
VK_TOKEN = "vk1.a.2bR6FTESYx6T-nJWKUB4yqZ73JtB7AIm9GV9Vq4TmSdSl4-hsStwxcfH90UQMocAP2UCm_FC8xzxnV9N7F5nsFywSlWz-Ut9OCOQ78Ml0qtXXB0KiV3zYY_Vcn-pfRfadtah_1jdghHqDgLKdAQDvMQJNM49ITzgxy0-CABmUyQyw1DVacUDTWyLpK0ES1rMpyPqeRFnj4Df3ftuO26A2A"



База данных

In [ ]:
import sqlite3
import json

class Database:
    def __init__(self, db_path="bot_database.db"):
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.cursor = self.conn.cursor()
        self._init_db()

    def _init_db(self):
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS users (
                user_id INTEGER PRIMARY KEY,
                state TEXT DEFAULT 'main_menu',
                current_subject TEXT DEFAULT NULL,
                subjects TEXT
            )
        ''')
        try:
            self.cursor.execute('SELECT temp_data FROM users LIMIT 1')
        except sqlite3.OperationalError:
            self.cursor.execute('ALTER TABLE users ADD COLUMN temp_data TEXT')
        self.conn.commit()

    def get_user(self, user_id):
        self.cursor.execute('SELECT state, current_subject, subjects, temp_data FROM users WHERE user_id = ?', (user_id,))
        row = self.cursor.fetchone()

        if row is None:
            self.cursor.execute('INSERT INTO users (user_id, subjects) VALUES (?, ?)', (user_id, '[]'))
            self.conn.commit()
            return 'main_menu', None, [], None

        subs = json.loads(row[2]) if row[2] else []
        return row[0], row[1], subs, row[3]

    def update_state(self, user_id, state):
        self.cursor.execute('UPDATE users SET state = ? WHERE user_id = ?', (state, user_id))
        self.conn.commit()

    def update_subject(self, user_id, subject):
        self.cursor.execute('UPDATE users SET current_subject = ? WHERE user_id = ?', (subject, user_id))
        self.conn.commit()

    def update_temp_data(self, user_id, data):
        self.cursor.execute('UPDATE users SET temp_data = ? WHERE user_id = ?', (data, user_id))
        self.conn.commit()

    def add_custom_subject(self, user_id, new_subject):
        _, _, subjects, _ = self.get_user(user_id)
        if new_subject not in subjects:
            subjects.append(new_subject)
            self.cursor.execute('UPDATE users SET subjects = ? WHERE user_id = ?', (json.dumps(subjects), user_id))
            self.conn.commit()
            return True
        return False


Google calendar (надо сделать)

In [ ]:
import os
import datetime
import re

class ObsidianAdapter:
    def __init__(self, folder_path):
        self.folder = folder_path

    def _sanitize_filename(self, name):
        """Удаляет запрещенные символы из названия файла"""
        return re.sub(r'[\\/*?:"<>|]', "", name).strip()

    def export_note(self, subject_name, title, text):
        date_str = datetime.datetime.now().strftime("%d.%m.%Y")
        safe_title = self._sanitize_filename(title)

        filename = f"[{subject_name}] {safe_title}.md"
        filepath = os.path.join(self.folder, filename)

        md_content = f"Название: {title}\nПредмет: {subject_name}\nДата: {date_str}\n\nКонспект:\n{text}"

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(md_content)

        return f"Заметка '{safe_title}' успешно сохранена!"

    def search_notes(self, subject_name, search_type="all", query=""):
        if not os.path.exists(self.folder):
            return "Папка на диске не найдена."

        files = [f for f in os.listdir(self.folder) if f.startswith(f"[{subject_name}]") and f.endswith('.md')]

        if not files:
            return f"Конспектов по предмету '{subject_name}' пока нет."

        results = []
        for f in files:
            filepath = os.path.join(self.folder, f)
            with open(filepath, 'r', encoding='utf-8') as file:
                content = file.read()

                if search_type == "title":
                    if query.lower() not in f.lower():
                        continue

                elif search_type == "date":
                    if query not in content:
                        continue

                clean_name = f.replace(f"[{subject_name}] ", "").replace(".md", "")

                date_match = re.search(r'Дата: (.*)', content)
                date_found = date_match.group(1) if date_match else "Неизвестно"

                if "Конспект:\n" in content:
                    preview = content.split("Конспект:\n")[1][:50].strip() + "..."
                else:
                    preview = ""

                results.append(f"📄 {clean_name} (от {date_found})\nТекст: {preview}\n")

        if not results:
            return f"По вашему запросу '{query}' ничего не найдено."

        return f"Найдено конспектов ({len(results)} шт.):\n\n" + "\n".join(results[:5]) + ("\n(Показаны первые 5)" if len(results) > 5 else "")

Интерфейс

In [ ]:
class VKView:
    def __init__(self, vk_api_method):
        self.vk = vk_api_method

    def send_message(self, user_id, text, keyboard=None):
        post = {'user_id': user_id, 'message': text, 'random_id': 0}
        if keyboard is not None:
            post['keyboard'] = keyboard
        self.vk.messages.send(**post)

    def get_main_menu(self):
        kb = VkKeyboard(one_time=False)
        kb.add_button('Выбрать предмет', color=VkKeyboardColor.PRIMARY)
        return kb.get_keyboard()

    def get_subjects_menu(self, user_subjects):
        kb = VkKeyboard(one_time=False)
        if not user_subjects:
            kb.add_button('Добавить предмет', color=VkKeyboardColor.POSITIVE)
        else:
            for i, subject in enumerate(user_subjects):
                kb.add_button(subject, color=VkKeyboardColor.PRIMARY)
                if (i + 1) % 2 == 0 and i != len(user_subjects) - 1:
                    kb.add_line()
            kb.add_line()
            kb.add_button('Добавить предмет', color=VkKeyboardColor.POSITIVE)
        kb.add_line()
        kb.add_button('Назад', color=VkKeyboardColor.NEGATIVE)
        return kb.get_keyboard()

    def get_subject_action_menu(self, subject):
        kb = VkKeyboard(one_time=False)
        kb.add_button('Поиск / Просмотр', color=VkKeyboardColor.POSITIVE)
        kb.add_line()
        kb.add_button('Написать конспект', color=VkKeyboardColor.PRIMARY)
        kb.add_line()
        kb.add_button('Назад к предметам', color=VkKeyboardColor.NEGATIVE)
        return kb.get_keyboard()

    def get_search_menu(self):
        kb = VkKeyboard(one_time=False)
        kb.add_button('По названию', color=VkKeyboardColor.PRIMARY)
        kb.add_button('По дате', color=VkKeyboardColor.PRIMARY)
        kb.add_line()
        kb.add_button('Все конспекты', color=VkKeyboardColor.SECONDARY)
        kb.add_line()
        kb.add_button('Отмена', color=VkKeyboardColor.NEGATIVE)
        return kb.get_keyboard()

    def get_cancel_menu(self):
        kb = VkKeyboard(one_time=False)
        kb.add_button('Отмена', color=VkKeyboardColor.NEGATIVE)
        return kb.get_keyboard()


бизинес

In [ ]:
class BotController:
    def __init__(self, db, view, obsidian_adapter):
        self.db = db
        self.view = view
        self.obsidian = obsidian_adapter

    def handle_message(self, user_id, text):
        text_lower = text.lower()
        state, current_subject, subjects, temp_data = self.db.get_user(user_id)

        if text_lower in ['отмена', 'назад', 'назад к предметам']:
            self.db.update_temp_data(user_id, None)

            if text_lower == 'назад к предметам':
                self.db.update_state(user_id, 'selecting_subject')
                self.view.send_message(user_id, "Ваши предметы:", self.view.get_subjects_menu(subjects))
            else:
                if state in ['search_menu', 'wait_search_title', 'wait_search_date', 'wait_note_title', 'wait_note_text']:
                    self.db.update_state(user_id, 'subject_menu')
                    self.view.send_message(user_id, "Действие отменено.", self.view.get_subject_action_menu(current_subject))
                else:
                    self.db.update_state(user_id, 'main_menu')
                    self.view.send_message(user_id, "Главное меню:", self.view.get_main_menu())
            return

        if state == 'main_menu':
            if text_lower in ['выбрать предмет', 'начать', 'start', 'привет']:
                self.db.update_state(user_id, 'selecting_subject')
                if not subjects:
                    self.view.send_message(user_id, "У вас пока нет предметов. Добавьте первый!", self.view.get_subjects_menu(subjects))
                else:
                    self.view.send_message(user_id, "Выберите предмет:", self.view.get_subjects_menu(subjects))
            else:
                self.view.send_message(user_id, "Воспользуйтесь кнопкой меню.", self.view.get_main_menu())

        elif state == 'selecting_subject':
            if text_lower == 'добавить предмет':
                self.db.update_state(user_id, 'wait_new_subject')
                self.view.send_message(user_id, "Введите название нового предмета:", self.view.get_cancel_menu())
            else:
                selected = next((s for s in subjects if s.lower() == text_lower), None)
                if selected:
                    self.db.update_subject(user_id, selected)
                    self.db.update_state(user_id, 'subject_menu')
                    self.view.send_message(user_id, f"Выбран предмет: {selected}", self.view.get_subject_action_menu(selected))
                else:
                    self.view.send_message(user_id, "Используйте кнопки.", self.view.get_subjects_menu(subjects))

        elif state == 'wait_new_subject':
            if self.db.add_custom_subject(user_id, text):
                self.view.send_message(user_id, f"Предмет '{text}' добавлен!")
            else:
                self.view.send_message(user_id, "Такой предмет уже существует.")

            _, _, new_subs, _ = self.db.get_user(user_id)
            self.db.update_state(user_id, 'selecting_subject')
            self.view.send_message(user_id, "Ваши предметы:", self.view.get_subjects_menu(new_subs))

        elif state == 'subject_menu':
            if text_lower == 'поиск / просмотр':
                self.db.update_state(user_id, 'search_menu')
                self.view.send_message(user_id, "Как будем искать?", self.view.get_search_menu())

            elif text_lower == 'написать конспект':
                self.db.update_state(user_id, 'wait_note_title')
                self.view.send_message(user_id, "Введите НАЗВАНИЕ конспекта (например: Лекция 1):", self.view.get_cancel_menu())
            else:
                self.view.send_message(user_id, "Используйте кнопки.", self.view.get_subject_action_menu(current_subject))

        elif state == 'wait_note_title':
            self.db.update_temp_data(user_id, text)
            self.db.update_state(user_id, 'wait_note_text')
            self.view.send_message(user_id, f"Название: '{text}'.\nТеперь отправьте ТЕКСТ конспекта:", self.view.get_cancel_menu())

        elif state == 'wait_note_text':
            title = temp_data if temp_data else "Без названия"
            res = self.obsidian.export_note(current_subject, title, text)

            self.db.update_temp_data(user_id, None)
            self.db.update_state(user_id, 'subject_menu')
            self.view.send_message(user_id, res, self.view.get_subject_action_menu(current_subject))

        elif state == 'search_menu':
            if text_lower == 'все конспекты':
                res = self.obsidian.search_notes(current_subject, search_type="all")
                self.db.update_state(user_id, 'subject_menu')
                self.view.send_message(user_id, res, self.view.get_subject_action_menu(current_subject))

            elif text_lower == 'по названию':
                self.db.update_state(user_id, 'wait_search_title')
                self.view.send_message(user_id, "Введите слово из названия конспекта:", self.view.get_cancel_menu())

            elif text_lower == 'по дате':
                self.db.update_state(user_id, 'wait_search_date')
                self.view.send_message(user_id, "Введите дату в формате ДД.ММ.ГГГГ (например, 25.10.2023):", self.view.get_cancel_menu())
            else:
                self.view.send_message(user_id, "Используйте кнопки.", self.view.get_search_menu())

        elif state == 'wait_search_title':
            res = self.obsidian.search_notes(current_subject, search_type="title", query=text)
            self.db.update_state(user_id, 'subject_menu')
            self.view.send_message(user_id, res, self.view.get_subject_action_menu(current_subject))

        elif state == 'wait_search_date':
            res = self.obsidian.search_notes(current_subject, search_type="date", query=text)
            self.db.update_state(user_id, 'subject_menu')
            self.view.send_message(user_id, res, self.view.get_subject_action_menu(current_subject))

Запуск

In [ ]:
try:
    vk_session = vk_api.VkApi(token=VK_TOKEN)
    vk = vk_session.get_api()
    longpoll = VkLongPoll(vk_session)

    db = Database()
    view = VKView(vk)
    obsidian_adapter = ObsidianAdapter(OBSIDIAN_FOLDER)

    controller = BotController(db, view, obsidian_adapter)

    print("Бот запущен.")

    for event in longpoll.listen():
        if event.type == VkEventType.MESSAGE_NEW and event.to_me:
            controller.handle_message(event.user_id, event.text)

except Exception as e:
    print(f"Ошибка при запуске: {e}")